# 03 — Carga y visualización de ECG desde VitalDB

Descarga de la señal ECG cruda para un subconjunto reducido de `case_id` usando la librería `vitaldb` y visualización de tramos.

**Advertencias**

- La descarga depende de conexión a internet y puede ser lenta.
- La señal se almacena localmente en `data/raw/vitaldb_waveforms/` (excluida del repositorio).
- Verifica el nombre del canal ECG real en VitalDB antes de descargar todos los casos.

## 1. Setup

In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.data_loading import load_metadata
from src.download import load_ecg_from_vitaldb, save_ecg_npy
from src.utils import ensure_dir, get_logger

logger = get_logger("nb03")

## 2. Selección del subconjunto de casos

Definir manualmente unos pocos `case_id` para probar la carga. Ajustar al criterio del usuario.

In [8]:
metadata = load_metadata()

# Tomar todos los case_id presentes en metadata.
# Se eliminan nulos y duplicados para evitar descargas repetidas.
all_case_ids = (
    metadata[config.CASE_ID_COLUMN]
    .dropna()
    .astype(int)
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Total de case_ids a cargar:", len(all_case_ids))
print("Primeros 10 case_ids:", all_case_ids[:10])

Total de case_ids a cargar: 482
Primeros 10 case_ids: [12, 13, 19, 42, 96, 98, 103, 110, 114, 146]


## 3. Descarga y persistencia local

In [9]:
# ensure_dir(config.VITALDB_WAVEFORMS_DIR)

# ecg_by_case = {}
# for cid in all_case_ids:
#     cached = config.VITALDB_WAVEFORMS_DIR / f"case_{cid}.npy"
#     if cached.exists():
#         logger.info("Cargando desde caché: %s", cached)
#         signal = np.load(cached)
#     else:
#         logger.info("Descargando desde VitalDB: case_id=%s", cid)
#         signal = load_ecg_from_vitaldb(
#             cid,
#             track_name=config.DEFAULT_ECG_TRACK_NAME,
#             sampling_rate_hz=config.DEFAULT_ECG_FS_HZ,
#         )
#         save_ecg_npy(signal, cid)
#     ecg_by_case[cid] = signal
#     print(f"case_id={cid} | shape={signal.shape}")

In [10]:
downloaded_files = list(config.VITALDB_WAVEFORMS_DIR.glob("case_*.npy"))

print("Archivos ECG descargados:", len(downloaded_files))
downloaded_files[:5]

Archivos ECG descargados: 482


[WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1001.npy'),
 WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1002.npy'),
 WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1018.npy'),
 WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1023.npy'),
 WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_103.npy')]

In [11]:
downloaded_case_ids = {
    int(p.stem.replace("case_", ""))
    for p in config.VITALDB_WAVEFORMS_DIR.glob("case_*.npy")
}

missing_case_ids = sorted(set(all_case_ids) - downloaded_case_ids)

print("Case_ids faltantes:", len(missing_case_ids))
print("Primeros faltantes:", missing_case_ids[:20])

Case_ids faltantes: 0
Primeros faltantes: []


## 4. Visualización de un tramo

In [12]:
fs = config.DEFAULT_ECG_FS_HZ
window_seconds = 10
max_cases_to_plot = 3

for i, (cid, signal) in enumerate(ecg_by_case.items()):
    if i >= max_cases_to_plot:
        break

    n = min(int(fs * window_seconds), signal.shape[0])
    t = np.arange(n) / fs

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(t, signal[:n], linewidth=0.8)
    ax.set_title(f"ECG | case_id={cid} | primeros {window_seconds} s")
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Amplitud")
    plt.tight_layout()
    plt.show()

NameError: name 'ecg_by_case' is not defined

## 5. Próximos pasos

- Continuar con `04_windowing_and_feature_engineering.ipynb` para construir ventanas alrededor de los latidos.